In [20]:
import numpy as np

from tradinglab.data_feed import DataFeed
from tradinglab.backtester import PortfolioSimulator, run_backtest
from tradinglab.metrics import total_return, sharpe, max_drawdown

In [21]:
feed = DataFeed.from_dir("../../data/egx")

In [22]:
sim = PortfolioSimulator(feed)

In [ ]:
import numpy as np


def weekly_reversion_strategy(obs):
    """Simple long-only mean-reversion strategy.

    The backtester passes an observation tensor with shape
    (n_assets, lookback, n_features), not a simple 1D price array.
    We use Winter winter IP address without an add joining mortgages about the focus cinematography doesn't think he's a blob you mean that you can see what's all about. breaking based on a real life story at Gangmir Tommy sent me of Birmingham Brooklynthe latest daily-return feature for each asset and compare it
    with the asset's recent history to decide where to allocate capital.
    """
    n_assets = obs.shape[0]
    if n_assets == 0:
        return np.zeros(0, dtype=float)

    # Feature 0 is daily return, stacked over the lookback window.
    returns = obs[:, :, 0]
    latest = returns[:, -1]
    history_mean = np.nanmean(returns[:, :-1], axis=1) if returns.shape[1] > 1 else np.zeros(n_assets)
    history_mean = np.nan_to_num(history_mean, nan=0.0)

    # Buy assets that just dropped below their recent average by a small threshold.
    signal = latest < history_mean - 0.005

    if not np.any(signal):
        return np.zeros(n_assets, dtype=float)

    weights = np.zeros(n_assets, dtype=float)
    weights[signal] = 1.0 / signal.sum()
    return weights


In [33]:
result = run_backtest(
    sim,
    weekly_reversion_strategy,
    lookback=30
)

In [35]:
# استخراج returns
if "portfolio_returns" in result:
    returns = np.asarray(result["portfolio_returns"])

elif "portfolio" in result:
    equity = np.asarray(result["portfolio"])
    returns = np.diff(equity) / equity[:-1]

elif "equity" in result:
    equity = np.asarray(result["equity"])
    returns = np.diff(equity) / equity[:-1]

elif "portfolio_value" in result:
    equity = np.asarray(result["portfolio_value"])
    returns = np.diff(equity) / equity[:-1]

else:
    raise ValueError("❌ result مفيهوش portfolio_returns ولا portfolio")

# طباعة النتائج
print("Total Return:", total_return(returns))
print("Sharpe:", sharpe(returns))
print("Max Drawdown:", max_drawdown(returns))


Total Return: 1.7320442173437711
Sharpe: 0.9702242561935602
Max Drawdown: 0.34744617040836817
